# V19 fix2 mnli: as fix1 (Kwon + K-prune c 0.5, 5 seeds, then d2i-RI 5 seeds) plus retry of a partial Drive checkpoint download and a published bl_stage.log. Tag v19-freeze-fix2; method unchanged

In [ ]:
%%bash
set -e
cd /kaggle/working && rm -rf quantum
git clone -q https://github.com/gourav-ai-engineer/quantum.git
cd quantum && git checkout -q v19-freeze-fix2 && git log --oneline -1 | tee /kaggle/working/commit.txt
mkdir -p /kaggle/working/fix2_mnli
export MODEL_NAME=distilbert-base-uncased BL_SKIP_CKPT=0 CK_REQUIRE=mnli BL_TASKS=mnli BL_CONS=0.5 PREP_ARGS=--no-ckpt
export CKPT_DIR_ROOT=/tmp/bl/ck_kprune/distilbert-base-uncased CKPT_SHA_FILE=$PWD/scripts/V19_CKPT_SHA256.json
bash scripts/v15_kaggle.sh bl /tmp/bl /kaggle/working/fix2_mnli > /kaggle/working/fix2_mnli/bl_stage.log 2>&1 || echo BL_FAILED
tail -5 /kaggle/working/fix2_mnli/bl_stage.log
python -c "import transformers,torch;assert transformers.__version__=='4.51.3',transformers.__version__;print(torch.__version__,transformers.__version__)" | tee /kaggle/working/env.txt
export BEAT_ARGS="--methods d2i_ri --tasks mnli --constraints 0.5 --seeds 7,42,77,123,2024 --kprune-masks /kaggle/working/fix2_mnli/bl/kprune_masks.npz --ckpt-root $CKPT_DIR_ROOT"
bash scripts/v15_kaggle.sh beat /tmp/bl /kaggle/working/fix2_mnli || echo BEAT_FAILED
cd /kaggle/working && tar czf fix2_mnli_results.tgz fix2_mnli commit.txt env.txt 2>/dev/null || true
ls -la /kaggle/working | head -20
